# BIST basket monthly rebalance — explained strategies v14

Bu sürüm önceki chart-first not defterini düzeltir.

En kritik düzeltme:
- **Equal**
- **HRP**
- **BL**
- **HYBRID**

stratejileri geri geldi. Yani artık sadece tek bir optimizer akışı yok; stratejileri yan yana karşılaştırabiliyorsun.

Amaç:
- geniş BIST sepeti + döviz + kıymetli madenler ile deney yapmak
- her ay yeni portföy kurmak
- stratejileri aynı veri üzerinde karşılaştırmak
- her grafik için neyi okuman gerektiğini açıkça yazmak

In [ ]:
!pip -q install yfinance PyPortfolioOpt plotly pandas numpy scipy scikit-learn openai

In [ ]:
import warnings, json, os
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
import plotly.express as px
import plotly.graph_objects as go

from pypfopt.risk_models import CovarianceShrinkage
from pypfopt.hierarchical_portfolio import HRPOpt
from pypfopt.black_litterman import BlackLittermanModel
from pypfopt.efficient_frontier import EfficientFrontier

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

## 1) Universe config

Bu hücrede yatırım evreni tanımlanır.

İçerik:
- BIST hisseleri
- USDTRY, EURTRY
- ALTIN_TRY, GUMUS_TRY, PLATIN_TRY

Dürüst not:
Aşağıdaki BIST listesi önceki geniş evren denemesinden alınmış başlangıç listesidir.
Bunu canlı doğrulanmış güncel resmi BIST100 listesi gibi sunmuyorum.

In [ ]:
BIST_TICKERS = ['AEFES.IS', 'AGHOL.IS', 'AKBNK.IS', 'AKFYE.IS', 'AKSA.IS', 'AKSEN.IS', 'ALARK.IS', 'ARCLK.IS', 'ASELS.IS', 'ASTOR.IS', 'BIMAS.IS', 'BOBET.IS', 'CCOLA.IS', 'CIMSA.IS', 'CWENE.IS', 'DOAS.IS', 'DOHOL.IS', 'ECILC.IS', 'EGEEN.IS', 'EKGYO.IS', 'ENERY.IS', 'ENJSA.IS', 'ENKAI.IS', 'EREGL.IS', 'EUPWR.IS', 'FROTO.IS', 'GARAN.IS', 'GESAN.IS', 'GUBRF.IS', 'HALKB.IS', 'HEKTS.IS', 'ISCTR.IS', 'ISMEN.IS', 'KAYSE.IS', 'KCAER.IS', 'KCHOL.IS', 'KLSER.IS', 'KOZAA.IS', 'KOZAL.IS', 'KRDMD.IS', 'MAVI.IS', 'MGROS.IS', 'MIATK.IS', 'ODAS.IS', 'OTKAR.IS', 'OYAKC.IS', 'PETKM.IS', 'PGSUS.IS', 'REEDR.IS', 'SASA.IS', 'SAHOL.IS', 'SDTTR.IS', 'SISE.IS', 'SKBNK.IS', 'SMRTG.IS', 'SOKM.IS', 'TAVHL.IS', 'TCELL.IS', 'THYAO.IS', 'TKFEN.IS', 'TOASO.IS', 'TSKB.IS', 'TTKOM.IS', 'TTRAK.IS', 'TUPRS.IS', 'ULKER.IS', 'VAKBN.IS', 'VESBE.IS', 'VESTL.IS', 'YKBNK.IS', 'ZOREN.IS']

FX_TICKERS = {'USDTRY': 'USDTRY=X', 'EURTRY': 'EURTRY=X'}
METAL_USD_TICKERS = {'ALTIN_USD': 'GC=F', 'GUMUS_USD': 'SI=F', 'PLATIN_USD': 'PL=F'}

CONFIG = {
    'salary_try': 25000,
    'target_test_months': 6,
    'target_lookback_days': 252,
    'download_period': '10y',
    'min_history_days': 180,
    'max_weight': 0.10,
    'gold_label': 'ALTIN_TRY',
    'gold_min': 0.05,
    'gold_max': 0.20,
    'hybrid_alpha': 0.25,
    'strategies': ['Equal', 'HRP', 'BL', 'HYBRID'],
}

print('BIST başlangıç evreni:', len(BIST_TICKERS))
print('Stratejiler:', CONFIG['strategies'])

## 2) Helper functions

Bu bölüm veri indirme, temizlik, format ve strateji yardımcılarını içerir.

Okurken önemli olan:
- burada sadece alt yapı var
- asıl yorum yapılacak bölümler daha aşağıda

In [ ]:
def try_fmt(x):
    if pd.isna(x):
        return '-'
    return f'₺{x:,.0f}'.replace(',', 'X').replace('.', ',').replace('X', '.')

def pct_fmt(x):
    if pd.isna(x):
        return '-'
    return f'{x*100:.2f}%'

def safe_asset_name(symbol):
    return symbol.replace('.IS', '').replace('=X', '').replace('=F', '')

def fetch_close_series(symbol, period):
    df = yf.download(symbol, period=period, auto_adjust=True, progress=False)
    if df is None or len(df) == 0:
        return None
    if isinstance(df.columns, pd.MultiIndex):
        level0 = list(df.columns.get_level_values(0))
        close = df.xs('Close', axis=1, level=0) if 'Close' in level0 else df.iloc[:, :1]
    else:
        close = df[['Close']] if 'Close' in df.columns else df.iloc[:, :1]
    if isinstance(close, pd.DataFrame):
        if close.shape[1] == 0:
            return None
        close = close.iloc[:, 0]
    s = pd.Series(close).dropna()
    s.name = symbol
    return s if len(s) else None

def normalize_prices(df):
    return df / df.iloc[0] * 100

def clean_returns(df):
    rets = df.pct_change().replace([np.inf, -np.inf], np.nan).dropna(how='any')
    if rets.empty:
        return rets
    valid_cols = [c for c in rets.columns if np.isfinite(rets[c]).all() and rets[c].std() > 1e-10]
    if len(valid_cols) == 0:
        return pd.DataFrame(index=rets.index)
    return rets[valid_cols]

def normalize_weights(w, index):
    w = pd.Series(w).reindex(index).fillna(0.0).astype(float)
    s = w.sum()
    if s <= 0 or not np.isfinite(s):
        return pd.Series(1/len(index), index=index)
    return w / s

def band_gold(weights, gold_label, gold_min, gold_max):
    w = weights.copy().astype(float)
    if gold_label not in w.index:
        return normalize_weights(w, w.index)
    g = float(w[gold_label])
    tg = min(max(g, gold_min), gold_max)
    if abs(tg - g) < 1e-12:
        return normalize_weights(w, w.index)
    rest = w.drop(gold_label)
    if rest.sum() <= 0:
        w[:] = 0.0
        w[gold_label] = 1.0
        return w
    rest = rest / rest.sum()
    w[gold_label] = tg
    w.loc[rest.index] = (1 - tg) * rest
    return normalize_weights(w, w.index)

def equal_weight(columns):
    return pd.Series(1 / len(columns), index=columns)

def inverse_vol_weight(price_window):
    rets = clean_returns(price_window)
    if rets.empty or rets.shape[1] == 0:
        return equal_weight(price_window.columns)
    iv = 1 / rets.std().replace(0, np.nan)
    iv = iv.replace([np.inf, -np.inf], np.nan).dropna()
    if iv.empty:
        return equal_weight(price_window.columns)
    w = pd.Series(0.0, index=price_window.columns)
    w.loc[iv.index] = iv / iv.sum()
    return normalize_weights(w, price_window.columns)

def hrp_weights_safe(price_window):
    rets = clean_returns(price_window)
    if rets.shape[0] < 2 or rets.shape[1] < 2:
        return inverse_vol_weight(price_window)
    try:
        hrp = HRPOpt(rets)
        w_sub = pd.Series(hrp.optimize()).reindex(rets.columns).fillna(0.0)
        w = pd.Series(0.0, index=price_window.columns)
        w.loc[w_sub.index] = w_sub
        return normalize_weights(w, price_window.columns)
    except Exception as e:
        print('HRP failed, inverse-vol kullanıldı:', str(e))
        return inverse_vol_weight(price_window)

def views_from_prices(price_window):
    rets = clean_returns(price_window)
    if rets.empty or rets.shape[1] == 0:
        return {c: 0.0 for c in price_window.columns}
    ann = rets.mean() * 252
    vol = rets.std() * np.sqrt(252)
    score = (ann / vol.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan).fillna(0)
    score = score.reindex(price_window.columns).fillna(0.0)
    return score.clip(-0.15, 0.15).to_dict()

def bl_weights_safe(price_window):
    rets = clean_returns(price_window)
    if rets.shape[0] < 2 or rets.shape[1] < 2:
        w = inverse_vol_weight(price_window)
        diag = pd.DataFrame({'asset': list(price_window.columns), 'view': [0.0]*len(price_window.columns), 'view_source': 'rules_insufficient_data'})
        return w, diag
    try:
        S = CovarianceShrinkage(rets, returns_data=True).ledoit_wolf()
        views = views_from_prices(price_window)
        bl = BlackLittermanModel(S, pi='equal', absolute_views=views, omega='default')
        post = bl.bl_returns()
        ef = EfficientFrontier(post, S, weight_bounds=(0, CONFIG['max_weight']))
        ef.max_sharpe(risk_free_rate=0.0)
        w = pd.Series(ef.clean_weights()).reindex(price_window.columns).fillna(0.0)
        w = normalize_weights(w, price_window.columns)
        w = band_gold(w, CONFIG['gold_label'], CONFIG['gold_min'], CONFIG['gold_max'])
        diag = pd.DataFrame({'asset': list(views.keys()), 'view': list(views.values()), 'view_source': 'rules'})
        return w, diag
    except Exception as e:
        print('BL failed, inverse-vol kullanıldı:', str(e))
        w = inverse_vol_weight(price_window)
        fallback_views = views_from_prices(price_window)
        diag = pd.DataFrame({'asset': list(fallback_views.keys()), 'view': list(fallback_views.values()), 'view_source': 'rules_bl_fallback'})
        return w, diag

def hybrid_weights(price_window):
    w_hrp = hrp_weights_safe(price_window)
    w_bl, diag = bl_weights_safe(price_window)
    a = CONFIG['hybrid_alpha']
    w = (1 - a) * w_hrp + a * w_bl
    w = normalize_weights(w, price_window.columns)
    w = w.clip(lower=0, upper=CONFIG['max_weight'])
    w = normalize_weights(w, price_window.columns)
    w = band_gold(w, CONFIG['gold_label'], CONFIG['gold_min'], CONFIG['gold_max'])
    return w, diag

def compute_weights(strategy_name, price_window):
    if strategy_name == 'Equal':
        return equal_weight(price_window.columns), None
    if strategy_name == 'HRP':
        return hrp_weights_safe(price_window), None
    if strategy_name == 'BL':
        return bl_weights_safe(price_window)
    if strategy_name == 'HYBRID':
        return hybrid_weights(price_window)
    raise ValueError(f'Unknown strategy: {strategy_name}')

## 3) Data coverage and eligibility

Bu bölüm en kritik güven kontrolüdür.

Bakman gereken şeyler:
- veri gerçekten inmiş mi
- hangi varlıklar testte kalmış
- ortak tarihçe yeterli mi

Eğer burada kalite zayıfsa, aşağıdaki bütün performans sonuçları anlamsız hale gelir.

In [ ]:
rows = []
series_list = []

for sym in BIST_TICKERS:
    s = fetch_close_series(sym, CONFIG['download_period'])
    rows.append({'source': sym, 'asset': safe_asset_name(sym), 'group': 'BIST', 'n': 0 if s is None else len(s), 'start': None if s is None else str(s.index.min().date()), 'end': None if s is None else str(s.index.max().date()), 'status': 'ok' if s is not None else 'empty'})
    if s is not None and len(s) >= CONFIG['min_history_days']:
        s = s.copy(); s.name = safe_asset_name(sym); series_list.append(s)

fx_series = {}
for label, sym in FX_TICKERS.items():
    s = fetch_close_series(sym, CONFIG['download_period'])
    fx_series[label] = s
    rows.append({'source': sym, 'asset': label, 'group': 'FX', 'n': 0 if s is None else len(s), 'start': None if s is None else str(s.index.min().date()), 'end': None if s is None else str(s.index.max().date()), 'status': 'ok' if s is not None else 'empty'})
    if s is not None and len(s) >= CONFIG['min_history_days']:
        s = s.copy(); s.name = label; series_list.append(s)

metal_usd = {}
for label, sym in METAL_USD_TICKERS.items():
    s = fetch_close_series(sym, CONFIG['download_period'])
    metal_usd[label] = s
    rows.append({'source': sym, 'asset': label, 'group': 'METAL_USD', 'n': 0 if s is None else len(s), 'start': None if s is None else str(s.index.min().date()), 'end': None if s is None else str(s.index.max().date()), 'status': 'ok' if s is not None else 'empty'})

usdtry = fx_series.get('USDTRY')
for try_label, usd_label in [('ALTIN_TRY', 'ALTIN_USD'), ('GUMUS_TRY', 'GUMUS_USD'), ('PLATIN_TRY', 'PLATIN_USD')]:
    base = metal_usd.get(usd_label)
    if base is not None and usdtry is not None:
        s = (base * usdtry).dropna()
        s.name = try_label
        rows.append({'source': f'{usd_label} * USDTRY', 'asset': try_label, 'group': 'METAL_TRY', 'n': len(s), 'start': str(s.index.min().date()), 'end': str(s.index.max().date()), 'status': 'ok'})
        if len(s) >= CONFIG['min_history_days']:
            series_list.append(s)

coverage = pd.DataFrame(rows)
display(coverage.head())

prices = pd.concat(series_list, axis=1).sort_index().ffill()
common_start = prices.apply(lambda s: s.first_valid_index()).max()
prices = prices.loc[common_start:].dropna()
returns = prices.pct_change().dropna()

eligible_assets = pd.DataFrame({'asset': prices.columns, 'group': [coverage.set_index('asset').loc[a, 'group'] if a in coverage['asset'].values else 'OTHER' for a in prices.columns]})

monthly_points = prices.resample('MS').first().dropna()
lookback = min(CONFIG['target_lookback_days'], max(60, len(prices) - 40))
months = CONFIG['target_test_months']

print('Ortak başlangıç:', common_start)
print('Kullanılan varlık sayısı:', prices.shape[1])
print('Toplam gün:', len(prices), 'lookback:', lookback, 'test_months:', months)
display(prices.tail())

### Coverage charts nasıl okunur?

İlk grafik:
- her grupta kaç seri indirilebildiğini gösterir

İkinci grafik:
- indirilenler içinden kaçının gerçekten teste kaldığını gösterir

İdeal durumda:
- `ok` yüksek olmalı
- testte kalan sayısı da anlamlı büyüklükte olmalı

In [ ]:
coverage_counts = coverage.groupby(['group', 'status']).size().reset_index(name='count')
fig = px.bar(coverage_counts, x='group', y='count', color='status', barmode='group', title='Veri kapsamı: grup bazında indirilen seriler')
fig.show()

eligible_counts = eligible_assets.groupby('group').size().reset_index(name='eligible_count')
fig = px.bar(eligible_counts, x='group', y='eligible_count', title='Teste kalan varlık sayısı')
fig.show()

## 4) Benchmark market charts

Burada strateji grafiği değil, önce piyasanın kendisi gösteriliyor.

Neden?
Çünkü portföy sonucunu yorumlamadan önce, temel rejimin ne olduğunu bilmek gerekir.

Göreceğin şeyler:
- BIST hisseleri genel olarak mı güçlüydü
- döviz mi baskındı
- altın / gümüş / platin ne yaptı
- bunlar birbirine ne kadar benziyordu

In [ ]:
bist_cols = [c for c in prices.columns if c not in ['USDTRY', 'EURTRY', 'ALTIN_TRY', 'GUMUS_TRY', 'PLATIN_TRY']]
bist_basket = (1 + returns[bist_cols].mean(axis=1)).cumprod() if len(bist_cols) else pd.Series(dtype=float)
if len(bist_basket):
    bist_basket.name = 'BIST_BASKET'
bench = pd.concat([bist_basket, prices[['USDTRY', 'EURTRY', 'ALTIN_TRY', 'GUMUS_TRY', 'PLATIN_TRY']]], axis=1).dropna()
bench_norm = normalize_prices(bench.tail(504))
fig = px.line(bench_norm, x=bench_norm.index, y=bench_norm.columns, title='Son ~2 yıl benchmark görünümü (normalize)')
fig.show()

bench_rets = bench.pct_change().dropna()
fig = px.imshow(bench_rets.corr(), text_auto=True, aspect='auto', zmin=-1, zmax=1, color_continuous_scale='RdBu_r', title='Benchmark korelasyon ısı haritası')
fig.show()

## 5) Strategy definitions

Burada tekrar görünür hale gelen stratejiler:

### Equal
Bütün varlıklara eşit ağırlık verir.  
En sade baseline budur.

### HRP
Hierarchical Risk Parity.  
Amaç riski daha dengeli dağıtmaktır. Pratikte bazen daha sakin görünen varlıklara yönelebilir.

### BL
Black-Litterman.  
Burada kural tabanlı view üretip bunları optimizasyona sokuyoruz.

### HYBRID
HRP ve BL arasında bir blend.  
Amaç sadece tek yönteme bağımlı kalmamak.

Aşağıdaki bütün strateji grafikleri artık bu dört yaklaşımı birlikte karşılaştırır.

## 6) Monthly walk-forward backtest

Bu bölüm gerçek hayata daha yakın olan kısımdır.

Akış:
- her ay maaş gibi sabit katkı eklenir
- sadece o tarihe kadar olan veri kullanılır
- strateji yeni portföy kurar
- bir sonraki rebalance tarihine kadar tutulur

Bu yüzden bu notebook statik tek portföy değil, aylık yeniden kurulan portföyleri gösterir.

In [ ]:
def get_rebalance_dates(prices, months):
    monthly = prices.resample('MS').first().dropna()
    month_starts = monthly.index[-months:]
    dates = []
    for dt in month_starts:
        idx = prices.index.searchsorted(dt)
        if idx < len(prices.index):
            dates.append(prices.index[idx])
    return pd.Index(pd.unique(pd.Index(dates)))

def run_strategy(strategy_name, prices, lookback, months):
    rebalance_dates = get_rebalance_dates(prices, months)
    cash = 0.0
    shares = pd.Series(0.0, index=prices.columns)
    hist, rebs, diags = [], [], []
    for i, dt in enumerate(rebalance_dates):
        cash += CONFIG['salary_try']
        window = prices.loc[:dt].tail(lookback)
        weights, diag = compute_weights(strategy_name, window)
        current = prices.loc[dt]
        total = float((shares * current).sum()) + cash
        shares = (weights * total) / current
        cash = 0.0
        nxt = rebalance_dates[i + 1] if i + 1 < len(rebalance_dates) else prices.index[-1]
        path = prices.loc[(prices.index >= dt) & (prices.index <= nxt)]
        vals = path.mul(shares, axis=1).sum(axis=1)
        for t, v in vals.items():
            hist.append({'date': t, 'strategy': strategy_name, 'portfolio_value': float(v)})
        row = {'date': dt, 'strategy': strategy_name, 'capital_after_contribution': total}
        for c, x in weights.items():
            row[f'weight_{c}'] = float(x)
        rebs.append(row)
        if diag is not None:
            tmp = diag.copy(); tmp['date'] = dt; tmp['strategy'] = strategy_name; diags.append(tmp)
    hist_df = pd.DataFrame(hist).drop_duplicates(['date', 'strategy'])
    weights_df = pd.DataFrame(rebs)
    diag_df = pd.concat(diags, ignore_index=True) if diags else pd.DataFrame()
    return hist_df, weights_df, diag_df

hist_parts, weight_parts, diag_parts = [], [], []
for s in CONFIG['strategies']:
    h, w, d = run_strategy(s, prices, lookback, months)
    hist_parts.append(h); weight_parts.append(w)
    if not d.empty:
        diag_parts.append(d)

hist_df = pd.concat(hist_parts, ignore_index=True)
weights_df = pd.concat(weight_parts, ignore_index=True)
diag_df = pd.concat(diag_parts, ignore_index=True) if diag_parts else pd.DataFrame()

display(hist_df.tail())
display(weights_df.tail())

## 7) Summary table

Bu tablo en üst seviye sonucu verir.

Nasıl okunur:
- **Contributed**: toplam yatırılan para
- **Ending Value**: test sonundaki portföy değeri
- **Net Profit**: kalan kazanç
- **TWR**: burada pratik karşılaştırma metriği gibi okunmalı
- **Ann Vol**: dalgalanma seviyesi
- **Sharpe**: getiri / risk verimliliği gibi okunur
- **MaxDD**: en kötü düşüş

Tek başına bir sütuna bakma.
Getiri, drawdown ve ağırlık davranışını birlikte okumak gerekir.

In [ ]:
rows = []
contributed = CONFIG['salary_try'] * months
for s in CONFIG['strategies']:
    sub = hist_df[hist_df['strategy'] == s].sort_values('date').copy()
    sub['ret'] = sub['portfolio_value'].pct_change().fillna(0.0)
    dd = sub['portfolio_value'] / sub['portfolio_value'].cummax() - 1
    rows.append({
        'Strategy': s,
        'Contributed': try_fmt(contributed),
        'Ending Value': try_fmt(sub['portfolio_value'].iloc[-1]),
        'Net Profit': try_fmt(sub['portfolio_value'].iloc[-1] - contributed),
        'TWR': pct_fmt(sub['portfolio_value'].iloc[-1] / contributed - 1),
        'Ann Vol': pct_fmt(sub['ret'].std() * np.sqrt(252)),
        'Sharpe': f"{(sub['ret'].mean() / (sub['ret'].std() + 1e-9) * np.sqrt(252)):.2f}",
        'MaxDD': pct_fmt(dd.min()),
    })
summary = pd.DataFrame(rows)
display(summary)

### Performance comparison charts nasıl okunur?

İlk çizgi grafik:
- stratejilerin portföy değerini yan yana gösterir

İkinci grafik:
- aynı stratejilerin zirveden ne kadar düştüğünü gösterir

Üçüncü grafik:
- ay bazında hangi stratejinin daha iyi / kötü davrandığını görmeyi kolaylaştırır

In [ ]:
fig = px.line(hist_df, x='date', y='portfolio_value', color='strategy', title='Strateji bazında portföy değeri')
fig.update_yaxes(tickprefix='₺', separatethousands=True)
fig.show()

dd_parts = []
for s in CONFIG['strategies']:
    sub = hist_df[hist_df['strategy'] == s].sort_values('date').copy()
    sub['drawdown'] = sub['portfolio_value'] / sub['portfolio_value'].cummax() - 1
    dd_parts.append(sub[['date', 'strategy', 'drawdown']])
dd_df = pd.concat(dd_parts, ignore_index=True)
fig = px.line(dd_df, x='date', y='drawdown', color='strategy', title='Strateji bazında drawdown')
fig.show()

mr_parts = []
for s in CONFIG['strategies']:
    sub = hist_df[hist_df['strategy'] == s].sort_values('date').copy().set_index('date')
    monthly = sub['portfolio_value'].resample('M').last().pct_change().dropna()
    mr_parts.append(pd.DataFrame({'date': monthly.index, 'strategy': s, 'monthly_return': monthly.values}))
mr_df = pd.concat(mr_parts, ignore_index=True)
fig = px.bar(mr_df, x='date', y='monthly_return', color='strategy', barmode='group', title='Aylık portföy getirileri')
fig.show()

## 8) Selection behavior charts

Bu bölüm stratejilerin gerçekte kaç varlık tuttuğunu ve hangi isimleri öne çıkardığını gösterir.

Neden önemli?
Çünkü bazen iyi görünen performans aslında çok dar ve riskli bir seçimin sonucudur.

In [ ]:
active_counts = []
for s in CONFIG['strategies']:
    sub = weights_df[weights_df['strategy'] == s].copy()
    weight_cols = [c for c in sub.columns if c.startswith('weight_')]
    sub['active_asset_count'] = (sub[weight_cols] > 0).sum(axis=1)
    active_counts.append(sub[['date', 'strategy', 'active_asset_count']])
active_counts_df = pd.concat(active_counts, ignore_index=True)

fig = px.bar(active_counts_df, x='date', y='active_asset_count', color='strategy', barmode='group', title='Her rebalance anında aktif varlık sayısı')
fig.show()

last_weights = weights_df.sort_values('date').groupby('strategy').tail(1).copy()
latest_long = last_weights.melt(id_vars=['date', 'strategy', 'capital_after_contribution'], var_name='asset', value_name='weight')
latest_long = latest_long[latest_long['asset'].str.startswith('weight_')].copy()
latest_long['asset'] = latest_long['asset'].str.replace('weight_', '', regex=False)
latest_long = latest_long[latest_long['weight'] > 0].copy()

fig = px.bar(latest_long.sort_values(['strategy', 'weight'], ascending=[True, False]),
             x='asset', y='weight', color='strategy', facet_row='strategy',
             title='Son rebalance anında strateji bazında ağırlıklar')
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
fig.show()

### Weight heatmap nasıl okunur?

Bu ısı haritası:
- zaman içinde hangi varlıkların öne çıktığını
- ağırlıkların sabit mi yoksa sık değişen mi olduğunu
- portföyün belirli birkaç isme yığılıp yığılmadığını

Okuma ipucu:
- koyu / yüksek hücreler sürekli aynı satırlarda toplanıyorsa, strateji daralıyor olabilir.

In [ ]:
weight_cols = [c for c in weights_df.columns if c.startswith('weight_')]
cumw = weights_df[weight_cols].sum().sort_values(ascending=False).head(15).index.tolist()

heat = weights_df[['date', 'strategy'] + cumw].copy()
heat['date'] = heat['date'].dt.strftime('%Y-%m-%d')
heat['row'] = heat['strategy'] + ' | ' + heat['date']
heat = heat.set_index('row')[cumw]
heat.columns = [c.replace('weight_', '') for c in heat.columns]

fig = px.imshow(heat.T, aspect='auto', title='Ağırlık evrimi ısı haritası (en çok kullanılan 15 ağırlık sütunu)')
fig.update_xaxes(title='Strateji | Rebalance tarihi')
fig.update_yaxes(title='Varlık')
fig.show()

## 9) BL / HYBRID views

Bu tablo özellikle BL ve HYBRID içinde hangi view'ların üretildiğini gösterir.

Nasıl okunur:
- sürekli tavana vuran pozitif view'lar varsa, view sistemi fazla kaba olabilir
- sürekli aynı varlıklar pozitif kalıyorsa, model çok tek temalı olabilir

In [ ]:
if len(diag_df):
    diag_show = diag_df.copy()
    diag_show['view_pct'] = diag_show['view'].map(pct_fmt)
    display(diag_show.tail(30))

## 10) Raw tables

Grafiklerden sonra detay tabloya bakmak istersen burada duruyor.

In [ ]:
out = last_weights[['strategy'] + [c for c in last_weights.columns if c.startswith('weight_')]].copy()
out.columns = ['Strategy'] + [c.replace('weight_', '') for c in out.columns[1:]]
for c in out.columns[1:]:
    out[c] = out[c].map(pct_fmt)
display(out.reset_index(drop=True))

print('Weights by rebalance:')
display(weights_df)

print('Portfolio value path:')
display(hist_df.tail(20))